# From Sight to Action — V0.1: extracting the Giant Fiber escape pathway

**Question:** which neurons bridge visual input to descending
movement-control neurons, in the MaleCNS connectome?

**Answer used for V0.1:** the *Giant Fiber visual escape circuit* — one of
the best-characterized sensorimotor pathways in *Drosophila*. Looming
(approaching-object) visual signals detected by lobula visual projection
neurons **LC4** and **LPLC2** drive **DNp01**, the Giant Fiber descending
neuron, which in turn drives **TTMn** (the tergotrochanteral "jump" motor
neuron) and **PSI** (which feeds the flight motor system). This matches
the circuit described in von Reyn et al. 2014/2017 and Ache et al. 2019 —
so this notebook is a *reproduction*, from real connectome data, of a
published finding, not a novel claim.

Data: MaleCNS v1.0 (minconf 0.5), public flat-connectome Feather files at
`gs://flyem-male-cns/v1.0/connectome-data/flat-connectome/` (CC-BY,
Janelia FlyEM). See `../README.md` for how these were downloaded.

In [1]:
import sys

sys.path.insert(0, "../src")

from sight_to_action.data import load_annotations, load_neurotransmitters, load_weights
from sight_to_action.pathway import build_pathway_graph, top_upstream_types
from sight_to_action.export import to_json

annotations = load_annotations()
neurotransmitters = load_neurotransmitters()
weights = load_weights()

print(f"{len(annotations):,} traced bodies")
print(f"{len(weights):,} directed connectivity edges (body-to-body)")

165,122 traced bodies
25,568,639 directed connectivity edges (body-to-body)


## 1. Confirm the circuit exists in the data

Search `body-annotations` for the known Giant Fiber circuit cell types by
their published names, and confirm their `superclass` matches what we'd
expect (visual projection neuron -> descending neuron -> motor neuron).

In [2]:
known_types = ["LC4", "LPLC2", "DNp01", "TTMn", "PSI"]
for t in known_types:
    sub = annotations[annotations["type"] == t]
    superclass = sub["superclass"].mode().iat[0] if len(sub) else "NOT FOUND"
    print(f"{t:8s}  n_bodies={len(sub):3d}  superclass={superclass}")

LC4       n_bodies=126  superclass=visual_projection
LPLC2     n_bodies=185  superclass=visual_projection
DNp01     n_bodies=  2  superclass=descending_neuron
TTMn      n_bodies=  2  superclass=vnc_motor
PSI       n_bodies=  2  superclass=vnc_efferent


## 2. Rank the strongest upstream inputs to the looming-detecting VPNs

Rather than hand-picking the "visual input" tier, rank every cell type
that synapses onto LC4/LPLC2 by total synaptic weight, and take the
strongest ones (excluding LC4/LPLC2's own recurrent connections to each
other).

In [3]:
ranked = top_upstream_types(weights, target_types=["LC4", "LPLC2"], n=15)
ranked = ranked[~ranked.index.isin(["LC4", "LPLC2"])]
ranked.head(10)

,total_weight,n_edges
type_pre,,
T2,46366,7375
TmY3,45917,5091
Tm4,38892,7407
Tm5Y,29372,4522
T5c,20789,7103
Tm2,20600,4183
T5d,19345,4962
T5b,19305,5926
Tm3,17494,4459


These are optic-lobe motion- and feature-detecting neurons — T4/T5
(direction-selective motion detectors) and Tm/T2/TmY-family medulla
neurons — exactly the classes of neuron known from the literature to
feed looming-detection VPNs.

In [4]:
visual_input_tier = ranked.head(10).index.tolist()
visual_input_tier

['T2', 'TmY3', 'Tm4', 'Tm5Y', 'T5c', 'Tm2', 'T5d', 'T5b', 'Tm3', 'T4c']

## 3. Build the type-level pathway graph

Four tiers, each strictly feed-forward into the next (edges that run
backward across tiers, e.g. descending -> visual, are dropped — this is
what turns the full recurrent connectome into one readable story):

1. Visual motion/feature detectors (optic lobe)
2. Looming-sensitive visual projection neurons (LC4, LPLC2)
3. The Giant Fiber descending neuron (DNp01)
4. Direct motor targets (TTMn, PSI)

In [5]:
tiers = [
    visual_input_tier,
    ["LC4", "LPLC2"],
    ["DNp01"],
    ["TTMn", "PSI"],
]

result = build_pathway_graph(
    weights, annotations, neurotransmitters, tiers, min_weight=3
)
print(f"{result.graph.number_of_nodes()} cell types, {result.graph.number_of_edges()} edges")
result.node_table

15 cell types, 118 edges


,type,tier,superclass,n_bodies,predicted_neurotransmitter
0,T2,0,ol_intrinsic,1630,acetylcholine
1,TmY3,0,ol_intrinsic,824,acetylcholine
2,Tm4,0,ol_intrinsic,1670,acetylcholine
3,Tm5Y,0,ol_intrinsic,898,acetylcholine
4,T5c,0,ol_intrinsic,1720,acetylcholine
5,Tm2,0,ol_intrinsic,1766,acetylcholine
6,T5d,0,ol_intrinsic,1620,acetylcholine
7,T5b,0,ol_intrinsic,1715,acetylcholine
8,Tm3,0,ol_intrinsic,2054,acetylcholine
9,T4c,0,ol_intrinsic,1778,acetylcholine


## 4. Sanity-check the core circuit edges

These four edges are the published backbone of the circuit — confirm
they survive the extraction with non-trivial synapse weight.

In [6]:
edge_lookup = {
    (r.type_pre, r.type_post): (r.weight, r.n_synapse_connections)
    for r in result.edge_table.itertuples()
}
for pair in [("LC4", "DNp01"), ("LPLC2", "DNp01"), ("DNp01", "TTMn"), ("DNp01", "PSI")]:
    print(pair, "-> weight, n_body_pairs =", edge_lookup.get(pair))

('LC4', 'DNp01') -> weight, n_body_pairs = (6362, 126)
('LPLC2', 'DNp01') -> weight, n_body_pairs = (4858, 182)
('DNp01', 'TTMn') -> weight, n_body_pairs = (90, 2)
('DNp01', 'PSI') -> weight, n_body_pairs = (12, 2)


## 5. Export for the web app

A small (~15 node) JSON file — the full 500MB+ raw tables never need to
ship to the browser.

In [7]:
out_path = to_json(
    result,
    "giant_fiber_pathway",
    meta={
        "title": "Visual looming input to the Giant Fiber escape circuit",
        "dataset": "MaleCNS v1.0 (minconf 0.5)",
        "source": "gs://flyem-male-cns/v1.0/connectome-data/flat-connectome/",
        "description": (
            "LC4/LPLC2 looming-sensitive visual projection neurons drive "
            "DNp01 (the Giant Fiber), which drives TTMn (jump muscle) and "
            "PSI (flight motor pathway)."
        ),
        "references": [
            "von Reyn et al. 2014, Neuron - 'A spike-timing mechanism for action selection'",
            "von Reyn et al. 2017, Nat Neurosci - 'Feature integration drives probabilistic behavior'",
            "Ache et al. 2019, Curr Biol - 'Neural basis for looming size and velocity encoding'",
        ],
    },
)
print("wrote", out_path)

wrote /Users/kiana/Desktop/Sight to Action/data/processed/giant_fiber_pathway.json
